# 03 · Prompt engineering as an experiment

Four prompt variants, one fixed harness, one table.

The "model" here is a **simulator** so the notebook runs offline in seconds: it emulates the
typical behaviour of a small instruct model — better definitions reduce adjacent-class
confusion, retrieved examples help most, and schema constraints remove parse failures without
changing accuracy. Replace `simulate_model` with a real 360M model and the harness is
unchanged.


In [ ]:
import numpy as np
import pandas as pd
from slm_decision import DecisionSpec, expected_cost, macro_f1

spec = DecisionSpec(
    name="ticket_routing",
    actions=["billing", "technical", "account", "review"],
    costs={("billing", "billing"): 0, ("billing", "technical"): 9, ("billing", "account"): 7,
           ("billing", "review"): 7, ("technical", "billing"): 9, ("technical", "technical"): 0,
           ("technical", "account"): 6, ("technical", "review"): 6, ("account", "billing"): 7,
           ("account", "technical"): 6, ("account", "account"): 0, ("account", "review"): 7,
           ("review", "billing"): 3, ("review", "technical"): 3, ("review", "account"): 3,
           ("review", "review"): 0},
    abstain="review",
)
rng = np.random.default_rng(11)

LABELS = ["billing", "technical", "account"]
gold = rng.choice(LABELS, 200, p=[0.45, 0.35, 0.20])
print(pd.Series(gold).value_counts(normalize=True).round(2).to_dict())

In [ ]:
CONFUSABLE = {"billing": ["account"], "technical": ["account"], "account": ["billing"]}

def simulate_model(gold_label, variant, rng):
    """Emulates a small instruct model's behaviour under four prompt variants."""
    p_correct = {"P0": 0.74, "P1": 0.79, "P2": 0.87, "P3": 0.87}[variant]
    p_abstain = {"P0": 0.08, "P1": 0.10, "P2": 0.11, "P3": 0.11}[variant]
    p_parse_fail = {"P0": 0.06, "P1": 0.06, "P2": 0.05, "P3": 0.00}[variant]

    r = rng.random()
    if r < p_parse_fail:
        return "review", False                      # fallback on unparseable output
    if r < p_parse_fail + p_abstain:
        return "review", True
    if r < p_parse_fail + p_abstain + p_correct:
        return gold_label, True
    return rng.choice(CONFUSABLE[gold_label]), True

In [ ]:
def run_variant(variant, gold, seed=0):
    r = np.random.default_rng(seed)
    rows = []
    for g in gold:
        action, parse_ok = simulate_model(g, variant, r)
        rows.append({"gold": g, "action": action, "parse_ok": parse_ok})
    return pd.DataFrame(rows)

def summarise(df):
    decided = df[df.action != "review"]
    return {
        "accuracy": round((df.action == df.gold).mean(), 3),
        "macro_f1": round(macro_f1(df.action, df.gold), 3),
        "parse_fail": round(1 - df.parse_ok.mean(), 3),
        "abstain_rate": round((df.action == "review").mean(), 3),
        "cost_per_case": round(expected_cost(spec, df.action, df.gold), 2),
    }

results = pd.DataFrame([{"variant": v, **summarise(run_variant(v, gold))}
                        for v in ["P0", "P1", "P2", "P3"]])
print(results.to_string(index=False))

## Is the difference real?

Paired comparison on the same examples needs a paired test. McNemar is three lines.


In [ ]:
from math import comb

def mcnemar(a_actions, b_actions, gold):
    a_right = np.asarray(a_actions) == np.asarray(gold)
    b_right = np.asarray(b_actions) == np.asarray(gold)
    b01 = int(np.sum(a_right & ~b_right))     # only A correct
    b10 = int(np.sum(~a_right & b_right))     # only B correct
    n = b01 + b10
    if n == 0:
        return 1.0
    # exact binomial two-sided p
    k = min(b01, b10)
    p = sum(comb(n, i) for i in range(k + 1)) / 2 ** n * 2
    return round(min(1.0, p), 4)

p1 = run_variant("P1", gold)["action"]
p2 = run_variant("P2", gold)["action"]
print("P2 vs P1  McNemar p =", mcnemar(p2, p1, gold), " n =", len(gold))

## Takeaways

- Examples beat wording: the jump from definitions (P1) to retrieved examples (P2) is the
  largest single change.
- A schema change (P3) removes parse failures and leaves accuracy untouched — exactly what a
  formatting change should do.
- Always report cost per case. It moves more than accuracy and it survives a budget review.
- 200 examples detects a ~5-point difference, not a 1-point one. Size the golden set against
  the effect you care about.
